# Advanced packages - MVR (Water Mover)

Part of the **synthetic-valley advanced-packages** series, which upgrades a calibrated base model's simple boundary conditions to MODFLOW 6 *advanced packages* one notebook at a time. Each notebook loads the shared advanced model from `models/` (creating it from the base model on first use with `notebook_helpers.load_or_create_advanced_model`) and adds one package, so they can be run independently and in any order - except where a package depends on others.

- **UZF** (`advanced-packages-uzf`) - recharge (RCH) -> Unsaturated Zone Flow
- **MAW** (`advanced-packages-maw`) - wells (WEL) -> Multi-Aquifer Well
- **SFR** (`advanced-packages-sfr`) - river (RIV) -> Streamflow Routing
- **LAK** (`advanced-packages-lak`) - high-K lake -> Lake package
- **MVR** (`advanced-packages-mvr`) - Water Mover (requires UZF, LAK, SFR) *(this notebook)*
- **Processing output** (`advanced-packages-processing`) - run the model and plot results

Add the Water Mover (MVR) package to route water between the advanced packages - here sending a fraction of each UZF cell's rejected infiltration to the lake or the stream network.

> **Dependencies.** The mover connects the UZF, LAK, and SFR packages, so all three must already exist in the advanced model. This notebook calls `notebook_helpers.require_packages` at the start and stops with an explanatory error if any are missing - run the UZF, LAK, and SFR notebooks first.

## Imports and setup

Import FloPy and the helpers, then choose the temporal resolution
(`sample_frequency`) and the model name.

In [ ]:
import flopy
from notebook_helpers import (
    drop_packages,
    load_or_create_advanced_model,
    require_packages,
)

In [ ]:
sample_frequency = "annual"  # monthly or annual
name = "sv"

## Load the advanced model and check dependencies

Load the shared advanced model and require that UZF, LAK, and SFR have already been built. If any is missing, `require_packages` raises with a message naming the notebook to run first.

In [ ]:
sim = load_or_create_advanced_model(sample_frequency, name)
gwf = sim.get_model(name)

require_packages(gwf, ["UZF-1", "LAK-1", "SFR-1"], "Water Mover (MVR)")
uzf = gwf.get_package("UZF-1")

## Build the MVR package

Move 10% of each UZF cell's available water to the lake (western half of the grid) or to the stream network (eastern half).

In [ ]:
packages = [["uzf-1"], ["LAK-1"], ["SFR-1"]]
perioddata = []
for values in uzf.packagedata.array:
    ifno, cellid = values["ifno"], values["cellid"]
    dst = "LAK-1" if cellid[1] < 20 else "SFR-1"
    perioddata.append(("uzf-1", int(ifno), dst, 0, "factor", 0.10))

In [ ]:
# add the mover (idempotent: drop any prior MVR first)
drop_packages(gwf, "mvr", "MVR")
mvr = flopy.mf6.ModflowGwfmvr(
    gwf,
    maxmvr=len(perioddata),
    maxpackages=len(packages),
    packages=packages,
    perioddata={0: perioddata},
)
gwf.get_package_list()

## Write and run

Write the updated advanced model back to `models/` and run it to confirm the mover-coupled model is valid.

In [ ]:
sim.write_simulation(silent=True)
success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"
print("MVR advanced model ran successfully")

## Recap

In this notebook you added the Water Mover (**MVR**) package, which routes a fraction of each UZF cell's available water to the lake (LAK) or the stream network (SFR), then wrote the updated advanced model back to `models/` and ran it. Because MVR connects UZF, LAK, and SFR, those three packages must already be built.